# Plot runs from `runs/`

Reads every `runs/*/config.json` + `metrics.csv`. Filter with config keys, e.g. `load_runs(dataset="mouse", training_mode="from_scratch")`; a filter value can also be a function. Metric throughout: mean per-neuron validation correlation. Figures are also saved to `research/run_figures/`.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from mouse_model.run_logging import load_runs, plot_runs, summary_table

FIG_DIR = os.path.join("research", "run_figures")
os.makedirs(FIG_DIR, exist_ok=True)
COLORS = {"mouse": "C0", "sensorium": "C1"}

runs = [r for r in load_runs() if r["summary"]]  # finished runs only
print(len(runs), "finished runs")
for row in summary_table(runs, keys=("dataset", "training_mode", "train_minutes_requested", "seed")):
    print(row)


def select(dataset, mode="from_scratch"):
    return [r for r in runs if r["config"].get("dataset") == dataset and r["config"].get("training_mode") == mode]


def best_cor(r):
    return r["summary"]["val_cor_at_best"]

## Validation correlation vs epoch, one panel per dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, dataset in zip(axes, ["mouse", "sensorium"]):
    sel = sorted(select(dataset), key=lambda r: r["config"]["split_info"]["train_minutes"])
    for r in sel:
        m = r["metrics"]
        ax.plot(m["epoch"], m["val_cor"], label=f"{r['config']['split_info']['train_minutes']:.1f} min")
    ax.set_title(f"{dataset}: from scratch")
    ax.set_xlabel("epoch (>= 100 batches each)")
    ax.grid(alpha=0.3)
    ax.legend(title="training data", fontsize=8)
axes[0].set_ylabel("val correlation")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "val_cor_vs_epoch.png"), dpi=120)

## Learning curve: correlation vs minutes of training data

Saturating fit \(r(m) = r_{max}\, m / (m + m_{50})\); then \(N_{90} = 9\, m_{50}\) minutes reaches 90% of \(r_{max}\). With one seed and few points this is a rough estimate.

In [ ]:
def sat(m, rmax, m50):
    return rmax * m / (m + m50)


fig, ax = plt.subplots(figsize=(7.5, 5))
fits = {}
for dataset in ["mouse", "sensorium"]:
    sel = select(dataset)
    if not sel:
        continue
    x = np.array([r["config"]["split_info"]["train_minutes"] for r in sel])
    y = np.array([best_cor(r) for r in sel])
    o = np.argsort(x)
    x, y = x[o], y[o]
    ax.plot(x, y, "o", color=COLORS[dataset], label=dataset)
    if len(x) >= 3:
        (rmax, m50), _ = curve_fit(sat, x, y, p0=[y.max(), x.mean()], bounds=([0, 1e-3], [1, 1e4]))
        fits[dataset] = dict(rmax=rmax, m50=m50, n90=9 * m50)
        xs = np.geomspace(x.min() * 0.8, max(x.max(), 9 * m50) * 1.2, 200)
        ax.plot(xs, sat(xs, rmax, m50), "-", color=COLORS[dataset], alpha=0.6,
                label=f"fit: r_max={rmax:.3f}, N90={9 * m50:.1f} min")
        ax.axvline(9 * m50, color=COLORS[dataset], ls=":", alpha=0.6)
ax.set_xscale("log")
ax.set_xlabel("training data (minutes of recording)")
ax.set_ylabel("val correlation at selected epoch (best val corr)")
ax.set_title("Data efficiency: free-moving mouse vs Sensorium (same CNN, scratch)")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "learning_curve_minutes.png"), dpi=120)
print(fits)

## Same curves vs gradient steps (separates data size from number of updates)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, dataset in zip(axes, ["mouse", "sensorium"]):
    for r in sorted(select(dataset), key=lambda r: r["config"]["split_info"]["train_minutes"]):
        m = r["metrics"]
        ax.plot(m["step"], m["val_cor"], label=f"{r['config']['split_info']['train_minutes']:.1f} min")
    ax.set_xscale("log")
    ax.set_xlabel("gradient steps")
    ax.set_title(dataset)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("val correlation")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "val_cor_vs_steps.png"), dpi=120)

## Transfer vs scratch (full training data)

In [ ]:
MODES = [("from_scratch", "scratch"), ("transfer_frozen_backbone", "frozen backbone"),
         ("transfer_finetune", "fine-tune")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (target, source) in zip(axes, [("mouse", "sensorium"), ("sensorium", "mouse")]):
    for i, (mode, label) in enumerate(MODES):
        sel = [r for r in select(target, mode) if r["config"].get("train_minutes_requested") is None]
        if not sel:
            continue
        r = sel[-1]
        m = r["metrics"]
        name = label if mode == "from_scratch" else f"{label} (from {source})"
        ax.plot(m["epoch"], m["val_cor"], color=f"C{i}", label=f"{name}: {best_cor(r):.3f}")
    ax.set_title(f"target: {target}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("val correlation")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transfer_vs_scratch.png"), dpi=120)

## Per-neuron correlation at the best epoch (full-data runs)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for r in runs:
    p = os.path.join(r["dir"], "per_neuron_best.npz")
    cfg = r["config"]
    if cfg.get("train_minutes_requested") is None and cfg.get("training_mode") == "from_scratch" and os.path.isfile(p):
        cor = np.load(p)["cor_per_neuron"]
        cor = cor[np.isfinite(cor)]
        ax.hist(cor, bins=40, alpha=0.5, density=True, color=COLORS[cfg["dataset"]],
                label=f"{cfg['dataset']} (n={len(cor)}, median {np.median(cor):.3f})")
ax.set_xlabel("per-neuron val correlation")
ax.set_ylabel("density")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "per_neuron_correlation.png"), dpi=120)

## Sensorium before vs after the input fix

Old curve: legacy pickle (image rows across time as "frames", random frame-level split). New: real frames, trial-level split.

In [ ]:
old_path = "epoch_vs_correlation_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_False.pkl"
fig, ax = plt.subplots(figsize=(7.5, 4.5))
if os.path.isfile(old_path):
    old = pickle.load(open(old_path, "rb"))
    old_train_n = 17917  # old random 70/30 split of 711 trials x 36 'frames'
    ax.plot((np.arange(len(old["val_cor_list"])) + 1) * old_train_n, old["val_cor_list"], "--", color="gray",
            label="old: wrong axis + frame-level split")
new = [r for r in select("sensorium") if r["config"].get("train_minutes_requested") is None]
if new:
    m = new[-1]["metrics"]
    ax.plot(m["samples_seen"], m["val_cor"], color=COLORS["sensorium"], label="new: real frames + trial-level split")
ax.set_xscale("log")
ax.set_xlabel("training samples seen")
ax.set_ylabel("val correlation")
ax.set_title("Sensorium CNN (scratch, full data)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "sensorium_before_after_fix.png"), dpi=120)